In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

from matplotlib.gridspec import GridSpec
#from ipywidgets import interact, FloatSlider, FloatLogSlider
import warnings
warnings.filterwarnings('ignore')
import os

from scipy.ndimage import gaussian_filter


import visualize_correct_z as vis
import matplotlib.colors as mcolors

from scipy.signal import find_peaks

import scipy.integrate as integrate


import glob
import re


# visualize paraview linecuts

In [ ]:
def parse_slice_filename(filename, sample):
    """
    Extracts slice_id, coord, and date from a filename of the form:
        {sample}_{slice_id}_{coord}_center_slice_{date}.csv

    Returns a dict with keys: slice_id, coord, date.
    Returns None if the filename doesn't match the expected pattern.
    """
    pattern = rf"^{re.escape(sample)}_(?P<slice_id>[^_]+)_(?P<coord>[xy])_center_slice_(?P<date>.+)\.csv$"
    match = re.match(pattern, filename)

    if match is None:
        return None

    return match.groupdict()


def get_slice_from_file(file_path, sample, coord, slice_id, date):
    """
    Loads and processes a single slice file (path already resolved).
    This is the same body as the old get_slice(), just decoupled from
    filename construction so it can be reused by the auto-discovery loop.
    """
    cfg = dict(sample=sample, coord=coord, slice_id=slice_id, date=date)

    if sample in ('665', '665_magconst'):
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 51
    elif sample in ('270', '270_magconst'):
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 66
    elif sample in ('452', '452_magconst', '452_orig'):
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 33
    elif sample in ('105', '105_magconst', '105_orig'):
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 45
    else:
        raise ValueError(f"Unrecognized sample '{sample}'")

    data = np.genfromtxt(file_path, delimiter=",", names=True)

    if coord == 'y':
        coord1 = data['Points0']
        coord2 = data['Points2']
        coord_name = data['Points1']
        mz = data['M2']
        mx = data['M0']
        my = data['M1']
    elif coord == 'x':
        coord1 = data['Points1']
        coord2 = data['Points2']
        coord_name = data['Points0']
        mz = data['M2']
        mx = data['M1']
        my = data['M0']

    mnorm = np.sqrt(mx**2 + my**2 + mz**2)
    mxn, myn, mzn = mx / mnorm, my / mnorm, mz / mnorm
    theta = np.arccos(np.clip(mzn, -1.0, 1.0))

    coord_label = round(float(np.mean(coord_name)), 3)

    cfg.update(
        slice=data, coord_name=coord_name, coord_label=coord_label,
        x=coord1, z=coord2, mx=mx, my=my, mz=mz,
        mxn=mxn, myn=myn, mzn=mzn, theta=theta, mnorm=mnorm,
    )
    return cfg


def load_sample_folder(sample, data_dir):
    folder = os.path.join(data_dir, sample)
    file_paths = sorted(glob.glob(os.path.join(folder, "*.csv")))

    print(f"Looking in: {os.path.abspath(folder)}")
    print(f"Found {len(file_paths)} files")

    result = {'x': {}, 'y': {}}

    for file_path in file_paths:
        filename = os.path.basename(file_path)
        parsed = parse_slice_filename(filename, sample)

        if parsed is None:
            print(f"Warning: skipping unrecognized filename '{filename}'")
            continue

        coord = parsed['coord']
        cfg = get_slice_from_file(
            file_path, sample, coord, parsed['slice_id'], parsed['date']
        )
        result[coord][cfg['coord_label']] = cfg

    return result


### User set up

In [ ]:
sample_ids = ['270', '270_magconst']
sample_colors = {'270': 'purple', '270_magconst': 'orange'}
# sample_colors = {'665': 'gold', '270': 'deeppink', '452': 'limegreen', '105': 'cyan'}
# sample_colors = {'665': 'gold', '270_magconst': 'deeppink', '452_magconst': 'limegreen', '105': 'cyan'}

data_dir = 'data_dir'  # top-level folder containing one subfolder per sample

samples = {sample: load_sample_folder(sample, data_dir) for sample in sample_ids}


field_pairs = [('mz', 'mzn'), ('my', 'myn'), ('mx', 'mxn')]

SMOOTH_DATA = True
SMOOTH_SIGMA = 2
how_many_slices = 9

for sample, coords in samples.items():
    for coord, slices_dict in coords.items():
        for label, data_dict in slices_dict.items():
            slices_dict[label] = vis.build_grids(
                data_dict, field_pairs, n_show=how_many_slices, xrange=None,
                smooth=False, sigma=SMOOTH_SIGMA
            )

for sample, coords in samples.items():
    for coord, slices_dict in coords.items():
        for slice_label, cfg in slices_dict.items():
            plot_grad = True
            vis.gradient(cfg, field_pairs, normalized=True, plot=False)

In [ ]:
# def find_tilt_angle(y, z):
#     slope, intercept = np.polyfit(y, z, 1)
#     angle = np.arctan(slope)
#     return angle

# def rotate_to_flat(y, z, angle=None):
#     """
#     Rotates a (y, z) point cloud to remove a linear tilt, so the data
#     becomes flat/horizontal. If angle is None, it's auto-detected via a
#     linear fit through the point cloud.
#     Returns y_flat, z_flat, and the angle used (in case you want to reuse
#     the same angle on a companion dataset, e.g. a different field slice
#     from the same tilted geometry).
#     """
#     if angle is None:
#         angle = find_tilt_angle(y, z)

#     y_center = y.mean()
#     z_center = z.mean()

#     y_shifted = y - y_center
#     z_shifted = z - z_center

#     cos_a, sin_a = np.cos(-angle), np.sin(-angle)
#     y_flat = y_shifted * cos_a - z_shifted * sin_a
#     z_flat = y_shifted * sin_a + z_shifted * cos_a

#     y_flat += y_center
#     z_flat += z_center

#     return y_flat, z_flat, angle



# for sample, coords in samples.items():
#     for coord, data_dict in coords.items():
#         x_arr = data_dict['x']
#         z_arr = data_dict['z']

#         calc_tilt = find_tilt_angle(x_arr, z_arr)
#         print(f'{sample} {coord}: tilt = {np.degrees(calc_tilt):.2f}°')

#         if abs(calc_tilt) > np.radians(1):  # only correct tilts over ~1 degree
#             x_flat, z_flat, angle = rotate_to_flat(x_arr, z_arr, angle=calc_tilt)
#             print(f"  Tilt angle removed: {np.degrees(angle):.2f}°")

#             # save the corrected coordinates back into the dictionary
#             data_dict['x_raw'] = x_arr  # preserve original, same pattern as your smoothing/_raw keys
#             data_dict['z_raw'] = z_arr
#             data_dict['x'] = x_flat
#             data_dict['z'] = z_flat
#             data_dict['tilt_angle'] = angle

#             plt.figure(figsize=(10, 5))
#             plt.scatter(x_flat, z_flat, s=1)
#             plt.xlabel('x')
#             plt.ylabel('z')
#             plt.title(f'{sample} {coord} — de-tilted slice ({np.degrees(angle):.1f}° removed)')
#             plt.show()
#         else:
#             data_dict['tilt_angle'] = 0.0

# full profile

In [ ]:
def plot_magnitude(cfg, z_indices=None):
    """
    Line-profile version: pick a handful of specific voxel rows (z_indices)
    to overlay, rather than showing the full heatmap.
    """
    unique_x = cfg['unique_x']
    z_grid = cfg['z_grid']

    if z_indices is None:
        # default: 5 evenly spaced rows across the available depth
        z_indices = np.linspace(0, cfg['n_rows'] - 1, 5).astype(int)

    colors = plt.cm.viridis(np.linspace(0, 1, len(z_indices)))

    fig, ax = plt.subplots(figsize=(49, 7))

    for c, row in zip(colors, z_indices):
        ax.plot(unique_x, cfg['M_mag'][row, :], color=c, lw=1.5,
                label=fr'z={z_grid[row]:.1f}')

    ax.set_ylabel(r'$M_{mag}$')
    ax.set_xlabel('x (pix)')
    ax.legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc='upper left')
    ax.set_title(
        f"Magnitude \n full {cfg['coord']}-slice profile \n "
        f"{cfg['sample']} (slice {cfg.get('coord_label', '?')})"
    )

    fig.tight_layout()
    plt.show()


# for sample, coords in samples.items():
#     for coord, slices_dict in coords.items():
#         for slice_label, cfg in slices_dict.items():
#             plot_magnitude(cfg)


# Segment and store domain walls 

In [ ]:
def segment_domain_walls_voxel(
    cfg,
    field='mzn_grid',
    grad_field=None,
    height=None,
    prominence=0.1,
    distance=None,
    plot=False,
    z_indices=None,
    offset=None,
    min_coverage=1.0,          # <-- now requires full-thickness presence by default
    max_shift_cols=3,
    max_gap_rows=1,
    require_contiguous=True,
):
    """
    Detect domain walls from peaks in |d(field)/dx| and track each physical
    wall through the z-stack by its x position.

    With min_coverage=1.0 (the default), a wall is only retained if it is
    detected in EVERY z-row (100% of the sample thickness). Lower values
    can still be used to loosen this for exploratory analysis, but partial
    walls are no longer the default behavior.

    Accepted walls are relabeled left-to-right by their mean x-position
    (leftmost = 0, rightmost = n) rather than by detection order.

    Returns
    -------
    walls : dict
        walls[wall_idx][row] = {...}  (wall_idx ordered left-to-right)
    flagged_walls : dict
        Diagnostic info for accepted-but-incomplete walls (only
        populated if min_coverage < 1.0).
    """

    if grad_field is None:
        grad_field = 'd' + field.replace('_grid', '') + '_dx'

    x_data = np.asarray(cfg['unique_x'])
    z_grid = np.asarray(cfg['z_grid'])

    profile_grid = np.asarray(cfg[field])
    grad_grid = np.asarray(cfg[grad_field])

    n_rows = profile_grid.shape[0]

    # ================================================================
    # 1. DETECT ALL GRADIENT PEAKS IN EACH Z ROW  (unchanged)
    # ================================================================

    detections_by_row = {}

    for row in range(n_rows):
        grad_row = grad_grid[row, :]
        prof_row = profile_grid[row, :]

        valid_mask = np.isfinite(grad_row)
        valid_idx = np.where(valid_mask)[0]

        if len(valid_idx) < 3:
            detections_by_row[row] = []
            continue

        grad_valid = grad_row[valid_idx]

        peaks, properties = find_peaks(
            np.abs(grad_valid),
            height=height,
            prominence=prominence,
            distance=distance,
        )

        if len(peaks) == 0:
            detections_by_row[row] = []
            continue

        peak_cols = np.sort(valid_idx[peaks])
        row_left_edge = int(valid_idx.min())
        row_right_edge = int(valid_idx.max())

        row_detections = []

        for peak_number, center_col in enumerate(peak_cols):
            center_col = int(center_col)

            if peak_number == 0:
                left_bound = row_left_edge
            else:
                left_bound = int((peak_cols[peak_number - 1] + center_col) // 2)

            if peak_number == len(peak_cols) - 1:
                right_bound = row_right_edge
            else:
                right_bound = int((center_col + peak_cols[peak_number + 1]) // 2)

            seg_slice = slice(left_bound, right_bound + 1)

            detection = {
                'x': x_data[seg_slice].copy(),
                'mz': prof_row[seg_slice].copy(),
                'grad': grad_row[seg_slice].copy(),
                'center_x': x_data[center_col],
                'center_col': center_col,
                'left_col': left_bound,
                'right_col': right_bound,
                'z_voxel': z_grid[row],
                'row': int(row),
            }

            row_detections.append(detection)

        detections_by_row[row] = row_detections

    # ================================================================
    # 2. TRACK PHYSICAL WALLS THROUGH DEPTH  (unchanged)
    # ================================================================

    tracks = {}
    next_track_id = 0

    for row in range(n_rows):
        detections = detections_by_row.get(row, [])
        if len(detections) == 0:
            continue

        active_tracks = []
        for track_id, track_rows in tracks.items():
            last_row = max(track_rows.keys())
            if row - last_row <= max_gap_rows + 1:
                last_detection = track_rows[last_row]
                active_tracks.append({
                    'track_id': track_id,
                    'last_row': last_row,
                    'center_col': last_detection['center_col'],
                })

        possible_matches = []
        for track_info in active_tracks:
            track_id = track_info['track_id']
            previous_col = track_info['center_col']

            for det_idx, detection in enumerate(detections):
                current_col = detection['center_col']
                dx_cols = abs(current_col - previous_col)

                if dx_cols <= max_shift_cols:
                    possible_matches.append((dx_cols, track_id, det_idx))

        possible_matches.sort(key=lambda item: item[0])

        used_tracks = set()
        used_detections = set()

        for dx_cols, track_id, det_idx in possible_matches:
            if track_id in used_tracks:
                continue
            if det_idx in used_detections:
                continue

            tracks[track_id][row] = detections[det_idx]
            used_tracks.add(track_id)
            used_detections.add(det_idx)

        for det_idx, detection in enumerate(detections):
            if det_idx in used_detections:
                continue
            tracks[next_track_id] = {row: detection}
            next_track_id += 1

    # ================================================================
    # 3. HELPER: LONGEST CONTIGUOUS DEPTH RUN  (unchanged)
    # ================================================================

    def longest_consecutive_run(rows):
        if len(rows) == 0:
            return 0
        rows = sorted(rows)
        longest = 1
        current = 1
        for previous_row, current_row in zip(rows[:-1], rows[1:]):
            if current_row == previous_row + 1:
                current += 1
                longest = max(longest, current)
            else:
                current = 1
        return longest

    # ================================================================
    # 4. FILTER WALLS ACCORDING TO DEPTH COVERAGE
    # ================================================================

    required_rows = int(np.ceil(min_coverage * n_rows))

    print(
        f"\nWall filtering: requiring at least "
        f"{required_rows}/{n_rows} rows "
        f"({min_coverage * 100:.0f}% of sample)"
    )

    accepted_candidates = []  # collected first, then sorted left-to-right

    for track_id, track_rows in tracks.items():
        rows_present = sorted(track_rows.keys())
        n_present = len(rows_present)
        coverage = n_present / n_rows
        longest_run = longest_consecutive_run(rows_present)
        contiguous_coverage = longest_run / n_rows

        passes_total = (n_present >= required_rows)
        passes_contiguous = (longest_run >= required_rows) if require_contiguous else True

        if not passes_total:
            print(
                f"DROPPING track {track_id}: {n_present}/{n_rows} rows "
                f"({coverage * 100:.1f}%)"
            )
            continue

        if not passes_contiguous:
            print(
                f"DROPPING track {track_id}: total coverage = "
                f"{coverage * 100:.1f}%, but longest continuous run = "
                f"{longest_run}/{n_rows} ({contiguous_coverage * 100:.1f}%)"
            )
            continue

        # representative x-position, used only for left-to-right ordering
        mean_center_x = float(np.mean([d['center_x'] for d in track_rows.values()]))

        accepted_candidates.append({
            'track_id': track_id,
            'track_rows': track_rows,
            'mean_center_x': mean_center_x,
            'n_present': n_present,
            'coverage': coverage,
            'rows_present': rows_present,
            'longest_run': longest_run,
            'contiguous_coverage': contiguous_coverage,
        })

    # ------------------------------------------------------------
    # Relabel left-to-right: leftmost wall = 0, rightmost = n
    # ------------------------------------------------------------
    accepted_candidates.sort(key=lambda c: c['mean_center_x'])

    walls = {}
    flagged_walls = {}

    for wall_idx, candidate in enumerate(accepted_candidates):
        walls[wall_idx] = candidate['track_rows']

        if candidate['n_present'] < n_rows:
            flagged_walls[wall_idx] = {
                'coverage': candidate['coverage'],
                'n_rows_present': candidate['n_present'],
                'n_rows_total': n_rows,
                'rows_present': candidate['rows_present'],
                'longest_contiguous_run': candidate['longest_run'],
                'contiguous_coverage': candidate['contiguous_coverage'],
                'original_track_id': candidate['track_id'],
            }
            print(
                f"FLAGGED wall {wall_idx} (x≈{candidate['mean_center_x']:.2f}): "
                f"{candidate['n_present']}/{n_rows} rows "
                f"({candidate['coverage'] * 100:.1f}%)"
            )
        else:
            print(
                f"KEEPING wall {wall_idx} (x≈{candidate['mean_center_x']:.2f}): "
                f"detected in all {n_rows} rows"
            )

    # ================================================================
    # 5. OPTIONAL PLOT
    # ================================================================

    if plot:
        plot_dw_segments(
            cfg, walls, field=field, grad_field=grad_field,
            z_indices=z_indices, offset=offset,
        )

    return walls, flagged_walls


def plot_dw_segments(cfg, walls, field='mzn_grid', grad_field=None, z_indices=None, offset=None):
    """
    Same stacked-offset line-cut style as before, with each detected
    domain wall's center marked as a short red dashed line, and now
    labeled with its left-to-right wall index (0 = leftmost).
    """
    x_data = cfg['unique_x']
    z_grid = cfg['z_grid']
    profile_grid = cfg[field]

    if grad_field is None:
        grad_field = 'd' + field.replace('_grid', '') + '_dx'
    grad_grid = cfg[grad_field]

    if offset is None:
        offset = 0.8 if 'n_grid' in field else 0.5

    if z_indices is None:
        z_indices = np.arange(cfg['n_rows'])
    colors = plt.cm.viridis(np.linspace(0, 1, len(z_indices)))

    fig, ax = plt.subplots(1, 2, figsize=(15, 7))

    # topmost row each wall appears in, used to place its index label once
    label_row_for_wall = {
        w_idx: max(w_rows.keys()) for w_idx, w_rows in walls.items()
    }

    for c, row in zip(colors, z_indices):
        shift = row * offset

        ax[0].plot(x_data, profile_grid[row, :] + shift, color=c, lw=1.5, marker='o')
        ax[1].plot(x_data, grad_grid[row, :] + shift, color=c, lw=1.5, marker='o')

        for a in ax:
            a.axhline(shift, color=c, lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'z voxel ={z_grid[row]:.1f}', xy=(0.22, shift - 0.4),
                       xycoords=('axes fraction', 'data'),
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=10, color=c)

        half_height = offset * 0.4
        for w_idx, w_rows in walls.items():
            if row in w_rows:
                center_x = w_rows[row]['center_x']
                for a in ax:
                    a.plot([center_x, center_x], [shift - half_height, shift + half_height],
                           color='red', linestyle='--', lw=1.5, zorder=4)

                # label the wall index once, at its topmost row
                if row == label_row_for_wall[w_idx]:
                    for a in ax:
                        a.annotate(f'DW {w_idx}', xy=(center_x, shift + half_height),
                                   xytext=(0, 4), textcoords='offset points',
                                   ha='center', va='bottom', fontsize=9,
                                   color='red', fontweight='bold')

    ax[0].set_ylabel(field)
    ax[0].set_xlabel('x (pix)')
    ax[0].set_title(f"{cfg['coord']} slice, {field}\n{cfg['sample']}")

    ax[1].set_xlabel('x (pix)')
    ax[1].set_ylabel(grad_field)
    ax[1].set_title(f"{cfg['coord']} slice, {grad_field}\n{cfg['sample']}")

    for a in ax:
        a.set_yticks([])

    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
PLOT_SEGMENTATION = False  # <-- flip to True to plot every sample/coord/slice

domain_walls = {}
flagged_by_sample = {}

for sample, coords in samples.items():
    domain_walls[sample] = {}
    flagged_by_sample[sample] = {}

    for coord, slices_dict in coords.items():
        domain_walls[sample][coord] = {}
        flagged_by_sample[sample][coord] = {}

        for slice_label, cfg in slices_dict.items():
            walls, flagged = segment_domain_walls_voxel(
                cfg, field='mzn_grid', prominence=0.1, plot=PLOT_SEGMENTATION
            )

            domain_walls[sample][coord][slice_label] = walls
            flagged_by_sample[sample][coord][slice_label] = flagged

            # also store directly on the slice's cfg dict, so downstream
            # functions that already receive cfg (e.g. area/inflection
            # analysis) have access to the wall segmentation too
            cfg['domain_walls'] = walls
            cfg['flagged_domain_walls'] = flagged

# Psi calculation (domain wall agnostic)

In [ ]:
def _aggregate_neel_bloch_across_slices(slices_dict, suffix):
    """
    Combines per-eta and overall Neel/Bloch stats across all slices for a
    single (sample, coord), assuming every slice shares the same n_rows
    (z-voxel count). Slices with a mismatched n_rows are skipped (with a
    warning) rather than breaking the averaging.

    Returns a dict shaped like a single-slice cfg would be, with:
        neel_pct_per_eta{suffix}, bloch_pct_per_eta{suffix}  (arrays, len n_rows)
        neel_pct_overall{suffix}, bloch_pct_overall{suffix}  (scalars)
        n_cols  (average n_cols across the slices actually used)
        n_slices_used
    computed as an UNWEIGHTED average across slices.
    """
    neel_per_eta_list, bloch_per_eta_list = [], []
    neel_overall_list, bloch_overall_list = [], []
    n_cols_list = []

    n_rows_ref = None

    for slice_label, cfg in slices_dict.items():
        neel_arr = np.asarray(cfg[f'neel_pct_per_eta{suffix}'])
        bloch_arr = np.asarray(cfg[f'bloch_pct_per_eta{suffix}'])

        if n_rows_ref is None:
            n_rows_ref = len(neel_arr)
        elif len(neel_arr) != n_rows_ref:
            print(f"Warning: slice '{slice_label}' has {len(neel_arr)} rows, "
                  f"expected {n_rows_ref}; skipping this slice in the aggregate.")
            continue

        neel_per_eta_list.append(neel_arr)
        bloch_per_eta_list.append(bloch_arr)
        neel_overall_list.append(cfg[f'neel_pct_overall{suffix}'])
        bloch_overall_list.append(cfg[f'bloch_pct_overall{suffix}'])
        n_cols_list.append(cfg['n_cols'])

    n_slices_used = len(neel_per_eta_list)

    if n_slices_used == 0:
        raise ValueError("No slices with matching n_rows were found to aggregate.")

    return {
        f'neel_pct_per_eta{suffix}': np.mean(neel_per_eta_list, axis=0),
        f'bloch_pct_per_eta{suffix}': np.mean(bloch_per_eta_list, axis=0),
        f'neel_pct_overall{suffix}': np.mean(neel_overall_list),
        f'bloch_pct_overall{suffix}': np.mean(bloch_overall_list),
        'n_cols': np.mean(n_cols_list),
        'n_slices_used': n_slices_used,
    }


def plot_neel_bloch_comparison(samples, sample_ids, normalized=True, offset=15):
    """
    Reproduces the original 4-panel (y per-eta / y overall bar / x per-eta /
    x overall bar) Neel-vs-Bloch summary figure, reading from either the
    normalized (default) or un-normalized branch of calc_neel_bloch_stats.

    Since each sample/coord now contains multiple slices, stats are
    averaged (unweighted) across all slices for that sample/coord before
    plotting -- so the figure still shows one line/bar per sample, just
    now representing an average over the slices collected for it.

    x-axis for the per-slice panels is a CENTERED voxel index (0 = the
    middle voxel row for that sample/coord), computed independently per
    sample/coord from its own n_rows.
    """
    suffix = '' if normalized else '_raw'
    tag = 'normalized' if normalized else 'un-normalized'

    fig, ax = plt.subplots(2, 2, figsize=(16, 14))

    overall_neel, overall_bloch = [], []
    overall_neel_err, overall_bloch_err = [], []
    overall_neelx, overall_blochx = [], []
    overall_neel_errx, overall_bloch_errx = [], []

    for s_idx, sample in enumerate(sample_ids):
        shift = s_idx * offset

        agg_by_coord = {
            coord: _aggregate_neel_bloch_across_slices(samples[sample][coord], suffix)
            for coord in ['y', 'x']
        }

        for coord, a in zip(['y', 'x'], [ax[0, 0], ax[1, 0]]):
            agg = agg_by_coord[coord]
            n_rows_cfg = len(agg[f'neel_pct_per_eta{suffix}'])
            z_indices = np.arange(n_rows_cfg) - (n_rows_cfg - 1) / 2  # 0 = center row
            n_cols = agg['n_cols']
            n_slices_used = agg['n_slices_used']

            # effective sample size across all pooled slices, for error bars
            n_eff = n_cols * n_slices_used

            p_neel = agg[f'neel_pct_per_eta{suffix}'] / 100
            p_bloch = agg[f'bloch_pct_per_eta{suffix}'] / 100
            neel_err = np.sqrt(p_neel * (1 - p_neel) / n_eff) * 100
            bloch_err = np.sqrt(p_bloch * (1 - p_bloch) / n_eff) * 100

            a.errorbar(z_indices, agg[f'neel_pct_per_eta{suffix}'] + shift, yerr=neel_err,
                       color='cyan', marker='o', lw=1.5, capsize=3)
            a.errorbar(z_indices, agg[f'bloch_pct_per_eta{suffix}'] + shift, yerr=bloch_err,
                       color='magenta', marker='o', lw=1.5, capsize=3)

        for coord, a in zip(['y', 'x'], [ax[0, 0], ax[1, 0]]):
            agg = agg_by_coord[coord]
            avg_plot = (agg[f'neel_pct_per_eta{suffix}'][0] + shift +
                        agg[f'bloch_pct_per_eta{suffix}'][0] + shift) / 2
            a.axhline(avg_plot, color='k', lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'{sample}', xy=(0.23, avg_plot + 1), xycoords=('axes fraction', 'data'),  ############ CHANGE LABEL POSITION HERE 
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=15, color='k')

        agg_y = agg_by_coord['y']
        n_total_y = agg_y['n_cols'] * len(agg_y[f'neel_pct_per_eta{suffix}']) * agg_y['n_slices_used']
        p_neel_all = agg_y[f'neel_pct_overall{suffix}'] / 100
        p_bloch_all = agg_y[f'bloch_pct_overall{suffix}'] / 100
        overall_neel.append(agg_y[f'neel_pct_overall{suffix}'])
        overall_bloch.append(agg_y[f'bloch_pct_overall{suffix}'])
        overall_neel_err.append(np.sqrt(p_neel_all * (1 - p_neel_all) / n_total_y) * 100)
        overall_bloch_err.append(np.sqrt(p_bloch_all * (1 - p_bloch_all) / n_total_y) * 100)

        agg_x = agg_by_coord['x']
        n_total_x = agg_x['n_cols'] * len(agg_x[f'neel_pct_per_eta{suffix}']) * agg_x['n_slices_used']
        p_neel_allx = agg_x[f'neel_pct_overall{suffix}'] / 100
        p_bloch_allx = agg_x[f'bloch_pct_overall{suffix}'] / 100
        overall_neelx.append(agg_x[f'neel_pct_overall{suffix}'])
        overall_blochx.append(agg_x[f'bloch_pct_overall{suffix}'])
        overall_neel_errx.append(np.sqrt(p_neel_allx * (1 - p_neel_allx) / n_total_x) * 100)
        overall_bloch_errx.append(np.sqrt(p_bloch_allx * (1 - p_bloch_allx) / n_total_x) * 100)

    ax[0, 0].set_title(f'y-slice ({tag})')
    ax[0, 0].set_xlabel('Voxel index (centered, 0 = middle)')
    ax[0, 0].set_ylabel('DW type (%)')

    color_handles = [
        plt.Line2D([0], [0], color='cyan', lw=2, label='% Néel'),
        plt.Line2D([0], [0], color='magenta', lw=2, label='% Bloch'),
    ]
    ax[0, 0].legend(handles=color_handles, fontsize=8, loc='upper left')

    x_pos = np.arange(len(sample_ids))
    width = 0.35
    ax[0, 1].bar(x_pos - width/2, overall_neel, width, yerr=overall_neel_err, capsize=4,
                 color='cyan', alpha=0.6, label='% Néel')
    ax[0, 1].bar(x_pos + width/2, overall_bloch, width, yerr=overall_bloch_err, capsize=4,
                 color='magenta', alpha=0.6, label='% Bloch')
    ax[0, 1].set_xticks(x_pos)
    ax[0, 1].set_xticklabels(sample_ids)
    ax[0, 1].set_xlabel('sample')
    ax[0, 1].set_ylabel('DW type (%)')
    ax[0, 1].set_title(f'Overall Néel/Bloch % per sample\ny-slice ({tag})')
    ax[0, 1].legend(fontsize=8)

    ax[1, 0].set_title(f'x-slice ({tag})')
    ax[1, 0].set_xlabel('Voxel index (centered, 0 = middle)')
    ax[1, 0].set_ylabel('DW type (%)')
    ax[1, 0].legend(handles=color_handles, fontsize=8, loc='upper left')

    ax[1, 1].bar(x_pos - width/2, overall_neelx, width, yerr=overall_neel_errx, capsize=4,
                 color='cyan', alpha=0.6, label='% Néel')
    ax[1, 1].bar(x_pos + width/2, overall_blochx, width, yerr=overall_bloch_errx, capsize=4,
                 color='magenta', alpha=0.6, label='% Bloch')
    ax[1, 1].set_xticks(x_pos)
    ax[1, 1].set_xticklabels(sample_ids)
    ax[1, 1].set_xlabel('sample')
    ax[1, 1].set_ylabel('DW type (%)')
    ax[1, 1].set_title(f'Overall Néel/Bloch % per sample\nx-slice ({tag})')
    ax[1, 1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
PLOT_PSI = False  # <-- flag: controls plotting for calc_psi AND plot_psi_bin_offset_w_highlight below

# -----------------------------------------------------------------------
# Step 1: calculate psi (azimuthal angle) per sample, coord, slice
# -----------------------------------------------------------------------
for sample, coords in samples.items():
    for coord, slices_dict in coords.items():
        for slice_label, cfg in slices_dict.items():
            vis.calc_psi(cfg, normalized=True, plot=False)


# -----------------------------------------------------------------------
# Step 2: Néel/Bloch stats + bin-offset plot per sample, coord, slice
# -----------------------------------------------------------------------
for sample, coords in samples.items():
    for coord, slices_dict in coords.items():
        for slice_label, cfg in slices_dict.items():
            vis.calc_neel_bloch_stats(cfg, normalized=True)
            # vis.calc_neel_bloch_stats(cfg, normalized=False)

            if PLOT_PSI:
                vis.plot_psi_bin_offset_w_highlight(
                    cfg, select_psi='', normalized=True, n_bins=11, offset=2.5
                )


# -----------------------------------------------------------------------
# Step 3: overall comparison across samples
# -----------------------------------------------------------------------
plot_neel_bloch_comparison(samples, sample_ids, normalized=True)

# Energy Calculations

In [ ]:

def find_inflection_point(x_data, curve, prominence=None):
    '''
    For a curve like (grad)**2 that does not cross zero, the inflection
    point of the underlying sigmoid corresponds to a local maximum of the
    curve (since |gradient| peaks at the inflection point).

    Finds all local maxima via scipy.signal.find_peaks, then selects the
    one closest to the center of x_data (handles cases where secondary
    peaks/shoulders exist away from the feature of interest).

    Sub-pixel location is refined via parabolic interpolation around the
    chosen peak.

    Returns np.nan if no local maximum is found.
    '''
    peaks, _ = find_peaks(curve, prominence=prominence)

    if len(peaks) == 0:
        return np.nan

    if len(peaks) > 1:
        center = np.mean(x_data)
        idx = peaks[np.argmin(np.abs(x_data[peaks] - center))]
    else:
        idx = peaks[0]

    x_peak = x_data[idx]

    # # parabolic sub-pixel refinement
    # if 0 < idx < len(curve) - 1:
    #     y0, y1, y2 = curve[idx - 1], curve[idx], curve[idx + 1]
    #     denom = (y0 - 2 * y1 + y2)
    #     if denom != 0:
    #         offset = 0.5 * (y0 - y2) / denom
    #     else:
    #         offset = 0.0
    #     dx = x_data[idx + 1] - x_data[idx]
    #     x_peak = x_data[idx] + offset * dx
    # else:
    #     x_peak = x_data[idx]

    return x_peak


def analyze_exchange_grid(cfg, normalized=True, z_indices=None):
    '''
    Computes the area under the curve (via np.trapezoid) and the inflection
    point (zero-crossing of the second derivative) for each z_voxel row of
    the exchange grid, and appends both results to cfg.
    Update the area method if trapezoidal approximation error is too high.
    '''

    area = []
    inflection = []

    suffix = '' if normalized else '_raw'
    energy_key = 'exchange_grid' if normalized else 'exchange_raw_grid'

    if z_indices is None:
        z_indices = np.arange(cfg['n_rows'])

    x_data = cfg['unique_x'] - cfg['unique_x'].mean()

    for j in z_indices:
        grid = cfg[energy_key][j, :]

        area_trap = integrate.trapezoid(grid, x=x_data)
        area.append(area_trap)

        x_inflect = find_inflection_point(x_data, grid)
        inflection.append(x_inflect)

    cfg[f'Exchange_area{suffix}'] = area
    cfg[f'Exchange_inflection{suffix}'] = inflection

    return cfg

In [ ]:
def extract_dw_energies(cfg, walls, normalized=True):
    """
    For each segmented domain wall in `walls`, slices exchange_grid,
    dmi_grid, and pma_grid (or their '_raw' counterparts) down to that
    wall's local x-range (left_col:right_col) at every row, and stores
    both the raw sliced segment and two summary stats per row: the
    trapezoidal area under the curve, and the mean value across the
    segment.

    Exchange also gets its peak/inflection location computed per row,
    same as before, since that has specific physical meaning here that
    DMI/PMA don't share.

    Results are stored in cfg[f'dw_energies{suffix}'][wall_idx] as a dict
    of arrays (one entry per row, ordered to match 'rows'):
        'rows', 'z_voxel',
        'exchange_segments' (list of 1D arrays), 'exchange_area', 'exchange_mean', 'exchange_inflection',
        'dmi_segments', 'dmi_area', 'dmi_mean',
        'pma_segments', 'pma_area', 'pma_mean',
    """
    suffix = '' if normalized else '_raw'
    energy_keys = {
        'exchange': f'exchange{suffix}_grid' if suffix else 'exchange_grid',
        'dmi': f'dmi{suffix}_grid' if suffix else 'dmi_grid',
        'pma': f'pma{suffix}_grid' if suffix else 'pma_grid',
    }
    # (handles the fact that raw keys are 'exchange_raw_grid' not 'exchange_grid_raw')
    if suffix:
        energy_keys = {
            'exchange': 'exchange_raw_grid',
            'dmi': 'dmi_raw_grid',
            'pma': 'pma_raw_grid',
        }

    x_data = np.asarray(cfg['unique_x'])
    grids = {name: np.asarray(cfg[key]) for name, key in energy_keys.items()}

    dw_summary = {}

    for wall_idx, row_dict in walls.items():
        rows_sorted = sorted(row_dict.keys())

        result = {
            'rows': rows_sorted,
            'z_voxel': [],
            'exchange_segments': [], 'exchange_area': [], 'exchange_mean': [], 'exchange_inflection': [],
            'dmi_segments': [], 'dmi_area': [], 'dmi_mean': [],
            'pma_segments': [], 'pma_area': [], 'pma_mean': [],
        }

        for row in rows_sorted:
            detection = row_dict[row]
            left_col = detection['left_col']
            right_col = detection['right_col']

            x_seg = x_data[left_col:right_col + 1]
            result['z_voxel'].append(detection['z_voxel'])

            for name, grid in grids.items():
                seg = grid[row, left_col:right_col + 1]
                area_trap = integrate.trapezoid(seg, x=x_seg)
                mean_val = np.nanmean(seg)

                result[f'{name}_segments'].append(seg)
                result[f'{name}_area'].append(area_trap)
                result[f'{name}_mean'].append(mean_val)

                if name == 'exchange':
                    x_inflect = find_inflection_point(x_seg, seg)
                    result['exchange_inflection'].append(x_inflect)

        # convert scalar lists to arrays for easy downstream aggregation
        for key in ('z_voxel', 'exchange_area', 'exchange_mean', 'exchange_inflection',
                    'dmi_area', 'dmi_mean', 'pma_area', 'pma_mean'):
            result[key] = np.array(result[key])

        dw_summary[wall_idx] = result

    cfg.setdefault(f'dw_energies{suffix}', {}).update(dw_summary)

    return dw_summary

In [ ]:
# 1. compute full-grid energies for every slice
for sample, coords in samples.items():
    for coord, slices_dict in coords.items():
        for slice_label, cfg in slices_dict.items():
            vis.calc_energies(cfg, coord, A=1.0, D=1.0, K=1.0)

# 2. (already done earlier) segment_domain_walls_voxel populated
#    cfg['domain_walls'] for every slice

# 3. extract per-wall energy segments + summary stats for every slice
for sample, coords in samples.items():
    for coord, slices_dict in coords.items():
        for slice_label, cfg in slices_dict.items():
            walls = cfg['domain_walls']
            extract_dw_energies(cfg, walls, normalized=True)

# 4. now aggregation/averaging across slices (per wall, per sample/coord)
#    can proceed using cfg['dw_energies'][wall_idx]['exchange_area'], etc.

In [ ]:
def analyze_exchange_grid_per_wall(cfg, walls, normalized=True):
    """
    For each segmented domain wall in `walls` (as returned by
    segment_domain_walls_voxel), computes the area under the exchange
    energy curve and the peak/inflection location, restricted to that
    wall's own local x-range (left_col:right_col) at each z-row -- rather
    than integrating across the full slice width.

    Since walls are now required to have 100% z-coverage, every wall
    should have exactly one entry per row.

    Results are stored two ways:
      1. Added directly onto each row's existing detection dict inside
         `walls` (safe -- these are just extra keys, not new row entries),
         as 'Exchange_area{suffix}' / 'Exchange_inflection{suffix}'.
      2. Collected into a per-wall summary dict returned by this function
         and also attached to cfg[f'dw_exchange{suffix}'][wall_idx], with
         keys 'rows', 'z_voxel', 'Exchange_area{suffix}',
         'Exchange_inflection{suffix}' -- ordered arrays, one value per
         row, for convenient downstream plotting/aggregation.
    """
    suffix = '' if normalized else '_raw'
    energy_key = 'exchange_grid' if normalized else 'exchange_raw_grid'

    x_data = np.asarray(cfg['unique_x'])
    grid = np.asarray(cfg[energy_key])

    dw_summary = {}

    for wall_idx, row_dict in walls.items():
        rows_sorted = sorted(row_dict.keys())

        area_arr, inflection_arr, z_arr = [], [], []

        for row in rows_sorted:
            detection = row_dict[row]
            left_col = detection['left_col']
            right_col = detection['right_col']

            x_seg = x_data[left_col:right_col + 1]
            grid_seg = grid[row, left_col:right_col + 1]

            area_trap = integrate.trapezoid(grid_seg, x=x_seg)
            x_inflect = find_inflection_point(x_seg, grid_seg)

            # (1) attach scalars directly onto the existing detection dict
            detection[f'Exchange_area{suffix}'] = area_trap
            detection[f'Exchange_inflection{suffix}'] = x_inflect

            area_arr.append(area_trap)
            inflection_arr.append(x_inflect)
            z_arr.append(detection['z_voxel'])

        # (2) per-wall summary, ordered by row
        dw_summary[wall_idx] = {
            'rows': rows_sorted,
            'z_voxel': np.array(z_arr),
            f'Exchange_area{suffix}': np.array(area_arr),
            f'Exchange_inflection{suffix}': np.array(inflection_arr),
        }

    cfg.setdefault(f'dw_exchange{suffix}', {}).update(dw_summary)

    return dw_summary

In [ ]:
for sample, coords in samples.items():
    for coord, slices_dict in coords.items():
        for slice_label, cfg in slices_dict.items():
            walls = cfg['domain_walls']  # stored earlier by segment_domain_walls_voxel
            analyze_exchange_grid_per_wall(cfg, walls, normalized=True)

In [ ]:
def _aggregate_exchange_per_wall_across_slices(slices_dict, suffix, wall_idx):
    """
    Combines per-z-voxel Exchange area and inflection-point values, for
    ONE specific domain-wall index, across all slices for a single
    (sample, coord). Slices that don't have this wall_idx (e.g. a
    different number of walls were detected in that crop) are skipped.
    """
    area_list, inflection_list = [], []
    z_grid_ref = None

    for slice_label, cfg in slices_dict.items():
        dw_key = f'dw_exchange{suffix}'
        if dw_key not in cfg or wall_idx not in cfg[dw_key]:
            continue

        wall_data = cfg[dw_key][wall_idx]
        z_vals = np.asarray(wall_data['z_voxel'])
        area_arr = np.asarray(wall_data[f'Exchange_area{suffix}'])
        inflection_arr = np.asarray(wall_data[f'Exchange_inflection{suffix}'])

        if z_grid_ref is None:
            z_grid_ref = z_vals
        elif len(z_vals) != len(z_grid_ref):
            print(f"Warning: slice '{slice_label}' wall {wall_idx} has {len(z_vals)} "
                  f"z-rows, expected {len(z_grid_ref)}; skipping.")
            continue

        area_list.append(area_arr)
        inflection_list.append(inflection_arr)

    n_slices_used = len(area_list)
    if n_slices_used == 0:
        return None

    area_stack = np.vstack(area_list)
    inflection_stack = np.vstack(inflection_list)

    mean_area = np.nanmean(area_stack, axis=0)
    mean_inflection = np.nanmean(inflection_stack, axis=0)

    if n_slices_used > 1:
        std_area = np.nanstd(area_stack, axis=0, ddof=1)
        std_inflection = np.nanstd(inflection_stack, axis=0, ddof=1)
    else:
        std_area = np.zeros_like(mean_area)
        std_inflection = np.zeros_like(mean_inflection)

    return {
        'z_grid': z_grid_ref,
        'mean_area': mean_area,
        'std_area': std_area,
        'mean_inflection': mean_inflection,
        'std_inflection': std_inflection,
        'n_slices_used': n_slices_used,
    }


def plot_exchange_area_and_inflection_per_wall(samples, normalized=True):
    '''
    For each coord, and for each domain-wall index (0 = leftmost),
    creates one figure with two subplots:
      (1) area vs z, for each sample (mean +/- stdev across slices)
      (2) inflection point location vs z, for each sample (mean +/-
          stdev across slices)

    Assumes analyze_exchange_grid_per_wall() has already been run on
    every slice's cfg dict, populating cfg[f'dw_exchange{suffix}'].
    '''
    suffix = '' if normalized else '_raw'

    palette = ['purple', 'orange', 'maroon']
    sample_names = list(samples.keys())
    sample_colors = {name: palette[i % len(palette)] for i, name in enumerate(sample_names)}

    thickness = [3, 3, 1.5]
    sample_thickness = {name: thickness[i % len(thickness)] for i, name in enumerate(sample_names)}

    all_coords = set()
    for sample, coords in samples.items():
        all_coords.update(coords.keys())

    figures = {}

    for coord in sorted(all_coords):

        # find the max number of wall indices present anywhere for this coord
        max_walls = 0
        for sample in sample_names:
            if coord not in samples[sample]:
                continue
            for slice_label, cfg in samples[sample][coord].items():
                n_walls = len(cfg.get(f'dw_exchange{suffix}', {}))
                max_walls = max(max_walls, n_walls)

        for wall_idx in range(max_walls):
            fig, ax = plt.subplots(2, 1, figsize=(10, 14))
            any_data = False

            for sample in sample_names:
                if coord not in samples[sample]:
                    continue

                agg = _aggregate_exchange_per_wall_across_slices(
                    samples[sample][coord], suffix, wall_idx
                )
                if agg is None:
                    continue

                any_data = True
                z_vals = agg['z_grid'] - agg['z_grid'][0]
                color = sample_colors[sample]
                thick = sample_thickness[sample]

                ax[0].errorbar(z_vals, agg['mean_area'], yerr=agg['std_area'],
                               color=color, label=sample, linewidth=thick,
                               marker='o', capsize=3)
                ax[1].errorbar(z_vals, agg['mean_inflection'], yerr=agg['std_inflection'],
                               color=color, label=sample, linewidth=thick,
                               marker='o', capsize=3)

            if not any_data:
                plt.close(fig)
                continue

            ax[0].set_xlabel('z voxel (bottom to top)')
            ax[0].set_ylabel('Area')
            ax[0].set_title(f'Area under Exchange vs z (mean \u00b1 stdev)\n'
                             f'{coord}-slice, domain wall {wall_idx}')
            ax[0].legend()

            ax[1].set_xlabel('z voxel (bottom to top)')
            ax[1].set_ylabel('Peak location')
            ax[1].set_title(f'Peak Location vs z (mean \u00b1 stdev)\n'
                             f'{coord}-slice, domain wall {wall_idx}')
            ax[1].legend()

            fig.tight_layout()
            figures[(coord, wall_idx)] = fig

    return figures


figs = plot_exchange_area_and_inflection_per_wall(samples, normalized=True)

In [ ]:
def _pool_dw_energies_across_walls_and_slices(slices_dict, suffix, quantity='exchange_area'):
    """
    Pools a given per-row quantity (e.g. 'exchange_area' or
    'exchange_inflection') across EVERY domain wall and EVERY slice for a
    single (sample, coord), into one flat pool per z-voxel row.

    Assumes all walls/slices share the same number of rows (n_rows), since
    walls are required to have 100% z-coverage. Mismatched lengths are
    skipped with a warning.

    Returns mean and sample stdev (ddof=1) per row, plus the z_grid used
    and how many (wall, slice) entries contributed to the pool overall.
    """
    dw_key = f'dw_energies{suffix}'

    pooled_values = []   # each entry: 1D array of length n_rows
    z_grid_ref = None

    for slice_label, cfg in slices_dict.items():
        if dw_key not in cfg:
            continue

        for wall_idx, wall_data in cfg[dw_key].items():
            values = np.asarray(wall_data[quantity])
            z_vals = np.asarray(wall_data['z_voxel'])

            if z_grid_ref is None:
                z_grid_ref = z_vals
            elif len(z_vals) != len(z_grid_ref):
                print(f"Warning: {slice_label}/wall {wall_idx} has {len(z_vals)} "
                      f"rows, expected {len(z_grid_ref)}; skipping.")
                continue

            pooled_values.append(values)

    n_pooled = len(pooled_values)
    if n_pooled == 0:
        return None

    stacked = np.vstack(pooled_values)  # shape (n_walls_and_slices, n_rows)

    mean_vals = np.nanmean(stacked, axis=0)
    std_vals = np.nanstd(stacked, axis=0, ddof=1) if n_pooled > 1 else np.zeros_like(mean_vals)

    return {
        'z_grid': z_grid_ref,
        'mean': mean_vals,
        'std': std_vals,
        'n_pooled': n_pooled,
    }


def plot_dw_energy_general_per_sample(samples, colors, normalized=True):
    """
    For each coord, creates one figure with two subplots:
      (1) average Exchange area vs z, per sample (mean +/- stdev, pooled
          across every domain wall and every slice)
      (2) average Exchange inflection/peak location vs z, per sample
          (mean +/- stdev, pooled the same way)
    """
    suffix = '' if normalized else '_raw'

    # palette = ['purple', 'orange', 'maroon']
    sample_names = list(samples.keys())
    # sample_colors = {name: palette[i % len(palette)] for i, name in enumerate(sample_names)}

    thickness = [2, 2, 1.5]
    sample_thickness = {name: thickness[i % len(thickness)] for i, name in enumerate(sample_names)}

    all_coords = set()
    for sample, coords in samples.items():
        all_coords.update(coords.keys())

    figures = {}

    for coord in sorted(all_coords):
        fig, ax = plt.subplots(2, 1, figsize=(10, 14))

        for sample in sample_names:
            if coord not in samples[sample]:
                continue

            slices_dict = samples[sample][coord]

            area_agg = _pool_dw_energies_across_walls_and_slices(
                slices_dict, suffix, quantity='exchange_area'
            )
            inflection_agg = _pool_dw_energies_across_walls_and_slices(
                slices_dict, suffix, quantity='exchange_inflection'
            )

            if area_agg is None or inflection_agg is None:
                print(f'Warning: {sample}/{coord} has no dw_energies data; skipping.')
                continue

            color = colors[sample]
            thick = sample_thickness[sample]

            z_vals = area_agg['z_grid'] - area_agg['z_grid'][0]

            ax[0].errorbar(z_vals, area_agg['mean'], yerr=area_agg['std'],
                           color=color, label=sample, linewidth=thick,
                           marker='o', capsize=3)
            ax[1].errorbar(z_vals, inflection_agg['mean'], yerr=inflection_agg['std'],
                           color=color, label=sample, linewidth=thick,
                           marker='o', capsize=3)

        ax[0].set_xlabel('z voxel (bottom to top)')
        ax[0].set_ylabel('Average Exchange Area')
        ax[0].set_title(f'Average Exchange Area vs z (\u00b1 stdev)\n{coord}-slice')
        ax[0].legend()

        ax[1].set_xlabel('z voxel (bottom to top)')
        ax[1].set_ylabel('Average Inflection Point Location')
        ax[1].set_title(f'Average Inflection Point vs z (\u00b1 stdev)\n{coord}-slice')
        ax[1].legend()

        fig.tight_layout()
        figures[coord] = fig

    return figures


figs = plot_dw_energy_general_per_sample(samples, sample_colors, normalized=True)

# Domain Wall width (hyperbolic tan)

In [ ]:
def fit_dw_wall(wall, sample, coord, wall_idx, offset = 1.5, label='', plot=True):
    """
    Fits a hyperbolic tangent to every z-voxel's segment for ONE domain wall
    (i.e. one entry of domain_walls[sample][coord]).

    Saves fit results in two places:
      - back into each row's segment dict (wall[row]['fit_params'], etc.) --
        so every voxel's fit lives right alongside its raw segment data
      - as return values (z_voxels, dw_width, dw_center arrays) -- for
        convenient use building a summary dict across walls/samples

    Rows are processed in sorted key order, NOT assumed to be 0..n-1
    contiguous, since some rows may be missing this wall.
    """
    pixel_size_nm = 8  # voxel size in nm

    rows_sorted = sorted(wall.keys())
    n = len(rows_sorted)
    colors_local = plt.cm.viridis(np.linspace(0, 1, n))

    z_voxels, dw_width, dw_center = [], [], []

    fig = plt.figure(figsize=(21, 6))
    fig.suptitle(f'Hyperbolic Tan Fit for {sample} {coord}, wall {wall_idx} ({label})',
                 fontsize=20, y=1.0)
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.set_xlabel('cross section (nm)')
    ax1.set_ylabel(r'$M_z$')
    ax1.set_title('Line cuts and fits')

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.set_xlabel('z voxel')
    ax2.set_ylabel('DW width')
    ax2.set_title('Fit DW width')

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.set_xlabel('z voxel')
    ax3.set_ylabel('DW center')
    ax3.set_title('Fit DW center')

    for j, row in enumerate(rows_sorted):
        seg = wall[row]
        x = seg['x']
        Mz = seg['mz']
        z_v = seg['z_voxel']

        shift = j * offset

        x_data = x - x.mean()
        x_nm = x_data * pixel_size_nm

        ax1.plot(x_nm, Mz + shift, color=colors_local[j], lw=1.5, marker='o', label=f'z={z_v:.1f}')

        hyperbolic_fit, hyperbolic_error = vis.fit_hyperbolic(x_nm, Mz)
        hyperbolic_plot = vis.hyperbolic(x_nm, *hyperbolic_fit)

        width = np.abs(hyperbolic_fit[2])
        center = hyperbolic_fit[1]

        ax1.plot(x_nm, hyperbolic_plot + shift, color=colors_local[j], lw=3, linestyle='--')

        # save fit results back into this row's own segment dict
        seg['fit_params'] = hyperbolic_fit
        seg['fit_error'] = hyperbolic_error
        seg['fit_x_nm'] = x_nm
        seg['fit_profile'] = hyperbolic_plot
        seg['dw_width'] = width
        seg['dw_center'] = center

        z_voxels.append(z_v)
        dw_width.append(width)
        dw_center.append(center)

    z_voxels = np.array(z_voxels)
    dw_width = np.array(dw_width)
    dw_center = np.array(dw_center)

    ax1.legend(fontsize=8, ncol=2, title='z voxel', loc='upper right')

    ax2.plot(z_voxels, dw_width, marker='o')
    ax2.axhline(np.mean(dw_width), color='red',
                label=f'mean \N{GREEK CAPITAL LETTER DELTA} = {np.mean(dw_width):.2f}')
    ax2.legend()

    ax3.plot(z_voxels, dw_center, marker='o')

    if not plot:
        plt.close(fig)
    else:
        plt.tight_layout()
        plt.show()

    return z_voxels, dw_width, dw_center

In [ ]:
PLOT_FITS = False  # <-- flag: controls whether fit_dw_wall generates a figure per wall/slice

dw_fits = {}

for sample, coords in samples.items():
    dw_fits[sample] = {}
    for coord, slices_dict in coords.items():
        dw_fits[sample][coord] = {}

        for slice_label, cfg in slices_dict.items():
            dw_fits[sample][coord][slice_label] = {}
            walls = cfg['domain_walls']

            for wall_idx, wall in walls.items():
                z_voxels, dw_width, dw_center = fit_dw_wall(
                    wall, sample, coord, wall_idx,
                    label=f'{coord}, slice {slice_label}', plot=PLOT_FITS
                )
                dw_fits[sample][coord][slice_label][wall_idx] = {
                    'z_voxel': z_voxels,
                    'dw_width': dw_width,
                    'dw_center': dw_center,
                }

In [ ]:
def average_dw_width_by_z_coord(dw_fits_for_coord, z_len, sample, coord):
    """
    Averages fitted domain-wall width and center over EVERY detected wall
    in EVERY slice, pooled together, at each z voxel.

    dw_fits_for_coord: dw_fits[sample][coord], structured as
        {slice_label: {wall_idx: {'z_voxel', 'dw_width', 'dw_center'}}}
    """
    mean_widths_list, std_widths_list = [], []
    mean_centers_list, std_centers_list = [], []

    for z in range(z_len):
        widths_list_perz = []
        centers_list_perz = []

        for slice_label, walls_dict in dw_fits_for_coord.items():
            for wall_idx, wall_data in walls_dict.items():
                dw_width_arr = wall_data['dw_width']
                dw_center_arr = wall_data['dw_center']

                if z >= len(dw_width_arr):
                    # this wall/slice doesn't reach this z-index (shouldn't
                    # normally happen since walls require 100% coverage,
                    # but different slices could still have different n_rows)
                    continue

                widths_list_perz.append(float(dw_width_arr[z]))
                centers_list_perz.append(float(dw_center_arr[z]))

        if len(widths_list_perz) == 0:
            mean_widths_list.append(np.nan)
            mean_centers_list.append(np.nan)
            std_widths_list.append(np.nan)
            std_centers_list.append(np.nan)
            continue

        mean_widths_list.append(np.mean(widths_list_perz))
        mean_centers_list.append(np.mean(centers_list_perz))
        std_widths_list.append(
            np.std(widths_list_perz, ddof=1) if len(widths_list_perz) > 1 else 0.0
        )
        std_centers_list.append(
            np.std(centers_list_perz, ddof=1) if len(centers_list_perz) > 1 else 0.0
        )

    return mean_widths_list, std_widths_list, mean_centers_list, std_centers_list


def plot_width_z_dependence(dw_averages, colors): 
    fig, ax = plt.subplots(2,1, figsize=(6, 12))


    for sample, coords in samples.items():
        print(f'sample = {sample}')

        for coord in coords:

            if coord == 'x': 
                line = ':'
            else: 
                line = '--'
            width = np.array(dw_averages[sample][coord]['width_z'])
            width_e = np.array(dw_averages[sample][coord]['width_ze'])
            center = np.array(dw_averages[sample][coord]['center_z'])
            center_e = np.array(dw_averages[sample][coord]['center_ze'])

            z = np.linspace(0, len(width), len(width))

            label = (f'{sample}, {coord}')

            ax[0].errorbar(z, width, yerr = width_e, color = colors[sample], marker='o', linestyle=line, linewidth=1.5, capsize=3, label=label)
            ax[1].errorbar(z, abs(center), yerr = center_e, color = colors[sample], marker='o', linestyle=line, linewidth=1.5, capsize=3, label=label)

    ax[0].set_xlabel('Depth through sample')
    ax[0].set_ylabel('Average domain wall width (nm)')
    ax[0].set_title('Average Domain Wall Width vs Depth')
    ax[0].legend()
    ax[0].set_ylim(-10,50)

    ax[1].set_xlabel('Depth through sample')
    ax[1].set_ylabel('Abs domain wall center')
    ax[1].set_title('Domain Wall Center vs Depth')
    ax[1].legend()

    plt.tight_layout()
    plt.show()

    return fig, ax

In [ ]:
dw_averages = {}

for sample, coords in samples.items():
    dw_averages[sample] = {}
    for coord, slices_dict in coords.items():

        n_rows_all = [len(cfg['z_grid']) for cfg in slices_dict.values()]
        if len(set(n_rows_all)) > 1:
            print(f"Warning: {sample}/{coord} slices have differing n_rows {n_rows_all}; "
                  f"using the max as z_len (missing z-indices will be skipped per-slice).")
        z_len = max(n_rows_all)

        average_width, error_width, average_center, error_center = average_dw_width_by_z_coord(
            dw_fits[sample][coord], z_len, sample, coord
        )
        dw_averages[sample][coord] = {
            'width_z': average_width,
            'width_ze': error_width,
            'center_z': average_center,
            'center_ze': error_center,
        }

In [ ]:
# sample_colors = {'665': 'gold', '270': 'deeppink', '452': 'limegreen', '105': 'cyan'}
# sample_colors = {'452': 'purple', '452_magconst': 'orange'}
plot_width_z_dependence(dw_averages, sample_colors)